# UMUD v3 — U-Net++ Pipeline

**Key fixes:** Scale clamping [0.025,0.15], U-Net++ dense skip connections, Tversky loss for fascicles, safe geometry FL=MT/sin(PA), TTA, per-resolution medians.

In [ ]:
# Cell 1: Imports & Paths
import os, math, warnings, cv2
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models as tvm
from scipy import ndimage
from PIL import Image
warnings.filterwarnings('ignore')

def seed_everything(s=42):
    np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True

seed_everything(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__}  Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

SEARCH_ROOTS = ['/kaggle/input', '/content', '.']

def walk_find(root, name, is_dir=True):
    for cur, dirs, files in os.walk(root):
        if name in (dirs if is_dir else files):
            return os.path.join(cur, name)
    return None

def find(name, is_dir=True):
    for r in SEARCH_ROOTS:
        if os.path.exists(r):
            p = walk_find(r, name, is_dir)
            if p: return p
    return None

APO_IMG_DIR  = find('apo_images_new_model_v1')
APO_MSK_DIR  = find('apo_masks_new_model_v1')
FASC_IMG_DIR = find('fasc_images_new_model_v1')
FASC_MSK_DIR = find('fasc_masks_new_model_v1')
TEST_IMG_DIR = find('test_set_v2')

SCALE_CSV = None
for r in SEARCH_ROOTS:
    if not os.path.exists(r): continue
    for cur, dirs, files in os.walk(r):
        for f in files:
            if 'universal_scale_metadata' in f.lower() and f.endswith('.csv'):
                SCALE_CSV = os.path.join(cur, f); break
        if SCALE_CSV: break

for k, v in [('APO_IMG',APO_IMG_DIR),('APO_MSK',APO_MSK_DIR),
             ('FASC_IMG',FASC_IMG_DIR),('FASC_MSK',FASC_MSK_DIR),
             ('TEST_IMG',TEST_IMG_DIR),('SCALE_CSV',SCALE_CSV)]:
    print(f'  [{k}] {v}')


In [ ]:
# Cell 2: Scale Lookup — clamp mm/px to [0.025, 0.15] to kill 10x OCR errors
# Root cause of soln1 bug: OCR read ruler as 0.65 mm/px instead of ~0.065 mm/px.
# That made FL 10x too large -> 34% of test images clamped at 130 mm ceiling.
MMPX_MIN, MMPX_MAX = 0.025, 0.15

def res_fallback_mmpx(w, h):
    if w == 1200 and h == 800:   return 50.0 / 720.0
    if w == 1640 and h == 1080:  return 45.0 / 980.0
    if w in (1088, 1069, 959):   return 45.0 / 600.0
    if 450 <= w <= 470:          return 45.0 / 460.0
    return 50.0 / max(h * 0.90, 1)

SCALE_MAP = {}; N_FROM_CSV = 0

if SCALE_CSV and os.path.exists(SCALE_CSV):
    sdf = pd.read_csv(SCALE_CSV)
    print(f'Scale CSV: {len(sdf)} rows')
    for _, row in sdf.iterrows():
        stem = str(row['record_id']).strip(); mmpx = None
        for col in ('fasc_mm_per_pixel', 'apo_mm_per_pixel'):
            v = row.get(col, float('nan'))
            if v == v and float(v) > 0:
                mmpx = float(v); break
        if mmpx is None:
            for col in ('fasc_spacing', 'apo_spacing'):
                sp = row.get(col, float('nan'))
                if sp == sp and float(sp) > 0:
                    mmpx = 10.0 / float(sp); break
        if mmpx is None:
            for col in ('fasc_depth_mm', 'apo_depth_mm'):
                d = row.get(col, float('nan'))
                if d == d and float(d) > 0:
                    mmpx = ('depth', float(d)); break
        if mmpx is not None:
            if isinstance(mmpx, float):
                SCALE_MAP[stem] = float(np.clip(mmpx, MMPX_MIN, MMPX_MAX)); N_FROM_CSV += 1
            else:
                SCALE_MAP[stem] = mmpx
    print(f'  Direct mmpx: {N_FROM_CSV}  Depth-only: {len(SCALE_MAP)-N_FROM_CSV}')
else:
    print('[WARN] Scale CSV not found.')

def get_mmpx(stem, orig_w, orig_h):
    val = SCALE_MAP.get(stem)
    if val is None:        mmpx = res_fallback_mmpx(orig_w, orig_h)
    elif isinstance(val, tuple): mmpx = val[1] / max(orig_h * 0.90, 1)
    else:                  mmpx = float(val)
    return float(np.clip(mmpx, MMPX_MIN, MMPX_MAX))

print('Scale lookup ready.')


In [ ]:
# Cell 3: Dataset Index
def image_files(folder):
    if not folder or not os.path.exists(folder): return set()
    return {f for f in os.listdir(folder)
            if not f.lower().startswith('thumbs') and
               os.path.splitext(f)[1].lower() in ('.tif', '.tiff', '.png', '.jpg')}

apo_imgs  = image_files(APO_IMG_DIR);  apo_msks  = image_files(APO_MSK_DIR)
fasc_imgs = image_files(FASC_IMG_DIR); fasc_msks = image_files(FASC_MSK_DIR)
test_imgs = image_files(TEST_IMG_DIR)

records = []
for fname in sorted(fasc_imgs):
    with Image.open(os.path.join(FASC_IMG_DIR, fname)) as im: w, h = im.size
    has_apo = fname in apo_msks; has_fasc = fname in fasc_msks
    records.append(dict(image_id=fname,
                        split='train_dual' if has_apo else 'train_fasc_only',
                        has_apo=has_apo, has_fasc=has_fasc, orig_w=w, orig_h=h))
for fname in sorted(test_imgs):
    with Image.open(os.path.join(TEST_IMG_DIR, fname)) as im: w, h = im.size
    records.append(dict(image_id=fname, split='test',
                        has_apo=False, has_fasc=False, orig_w=w, orig_h=h))

meta = pd.DataFrame(records)
print(meta['split'].value_counts().to_string())
print('\nTest resolution breakdown:')
print(meta[meta['split']=='test'].groupby(['orig_w','orig_h'])['image_id'].count())


In [ ]:
# Cell 4: Data Loading & Augmentation
IMG_H, IMG_W = 384, 576

def load_img_np(path):
    img = np.array(Image.open(path).convert('L'), dtype=np.float32) / 255.0
    return cv2.resize(img, (IMG_W, IMG_H), interpolation=cv2.INTER_LINEAR)

def load_mask_np(path):
    m = np.array(Image.open(path).convert('L'), dtype=np.uint8)
    if (m > 128).mean() > 0.5:
        m = (m == 0).astype(np.uint8) * 255
    m = cv2.resize(m, (IMG_W, IMG_H), interpolation=cv2.INTER_AREA)
    return (m > 64).astype(np.float32)

def augment(img_np, apo_np, fasc_np):
    if np.random.rand() < 0.5:
        img_np=img_np[:,::-1].copy(); apo_np=apo_np[:,::-1].copy(); fasc_np=fasc_np[:,::-1].copy()
    angle = np.random.uniform(-7, 7)
    M = cv2.getRotationMatrix2D((IMG_W/2, IMG_H/2), angle, 1.0)
    img_np  = cv2.warpAffine(img_np,  M,(IMG_W,IMG_H),flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    apo_np  = cv2.warpAffine(apo_np,  M,(IMG_W,IMG_H),flags=cv2.INTER_NEAREST,borderMode=cv2.BORDER_CONSTANT)
    fasc_np = cv2.warpAffine(fasc_np, M,(IMG_W,IMG_H),flags=cv2.INTER_NEAREST,borderMode=cv2.BORDER_CONSTANT)
    gamma = np.random.uniform(0.7, 1.4)
    img_np = np.clip(img_np**gamma, 0, 1)
    img_np = np.clip(img_np*np.random.uniform(0.8,1.2)+np.random.uniform(-0.08,0.08),0,1)
    img_np += np.random.normal(0, np.random.uniform(0, 0.04), img_np.shape).astype(np.float32)
    return np.clip(img_np,0,1).astype(np.float32), apo_np.astype(np.float32), fasc_np.astype(np.float32)

class MuscleDataset(Dataset):
    def __init__(self, df, aug=False): self.df=df.reset_index(drop=True); self.aug=aug
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]; fname=row['image_id']; is_test=row['split']=='test'
        img_dir=TEST_IMG_DIR if is_test else FASC_IMG_DIR
        img_np=load_img_np(os.path.join(img_dir,fname))
        if is_test:
            return torch.tensor(img_np).unsqueeze(0),fname,int(row['orig_w']),int(row['orig_h'])
        zero=np.zeros((IMG_H,IMG_W),np.float32)
        apo_np =load_mask_np(os.path.join(APO_MSK_DIR, fname)) if row['has_apo']  and APO_MSK_DIR  else zero
        fasc_np=load_mask_np(os.path.join(FASC_MSK_DIR,fname)) if row['has_fasc'] and FASC_MSK_DIR else zero
        if self.aug: img_np,apo_np,fasc_np=augment(img_np,apo_np,fasc_np)
        masks=torch.tensor(np.stack([apo_np,fasc_np],0),dtype=torch.float32)
        flags=torch.tensor([float(row['has_apo']),float(row['has_fasc'])],dtype=torch.float32)
        return torch.tensor(img_np).unsqueeze(0),masks,flags

print('Dataset ready.')


In [ ]:
# Cell 5: U-Net++ Architecture
class ConvBnRelu(nn.Module):
    def __init__(self, ci, co):
        super().__init__()
        self.m=nn.Sequential(nn.Conv2d(ci,co,3,padding=1,bias=False),nn.BatchNorm2d(co),nn.ReLU(True))
    def forward(self,x): return self.m(x)

class DenseDecodeNode(nn.Module):
    def __init__(self, up_in_c, skip_total_c, out_c):
        super().__init__()
        self.up=nn.ConvTranspose2d(up_in_c,out_c,2,stride=2)
        self.conv=nn.Sequential(ConvBnRelu(out_c+skip_total_c,out_c),ConvBnRelu(out_c,out_c))
    def forward(self, x_below, *skips):
        x=self.up(x_below)
        if skips:
            ref=skips[0]
            if x.shape[2:]!=ref.shape[2:]:
                x=F.interpolate(x,size=ref.shape[2:],mode='bilinear',align_corners=False)
            x=torch.cat([x]+list(skips),dim=1)
        return self.conv(x)

class UNetPlusPlus(nn.Module):
    def __init__(self, deep_supervision=True):
        super().__init__(); self.ds=deep_supervision
        enc=tvm.resnet34(weights=tvm.ResNet34_Weights.DEFAULT)
        w0=enc.conv1.weight.data.mean(dim=1,keepdim=True)
        enc.conv1=nn.Conv2d(1,64,7,stride=2,padding=3,bias=False)
        enc.conv1.weight.data=w0
        self.e0=nn.Sequential(enc.conv1,enc.bn1,enc.relu)
        self.pool=enc.maxpool
        self.e1=enc.layer1; self.e2=enc.layer2; self.e3=enc.layer3; self.e4=enc.layer4
        self.d3_0=DenseDecodeNode(512,256,256)
        self.d2_0=DenseDecodeNode(256,128,128);    self.d2_1=DenseDecodeNode(256,128+128,128)
        self.d1_0=DenseDecodeNode(128,64,64);      self.d1_1=DenseDecodeNode(128,64+64,64)
        self.d1_2=DenseDecodeNode(128,64+64+64,64)
        self.d0_0=DenseDecodeNode(64,64,32);       self.d0_1=DenseDecodeNode(64,64+32,32)
        self.d0_2=DenseDecodeNode(64,64+32+32,32); self.d0_3=DenseDecodeNode(64,64+32+32+32,32)
        self.up_final=nn.ConvTranspose2d(32,32,2,stride=2)
        self.head_ds1=nn.Conv2d(128,2,1); self.head_ds2=nn.Conv2d(64,2,1)
        self.head_ds3=nn.Conv2d(32,2,1);  self.head_out=nn.Conv2d(32,2,1)

    def forward(self,x):
        e0=self.e0(x); ep=self.pool(e0)
        e1=self.e1(ep); e2=self.e2(e1); e3=self.e3(e2); e4=self.e4(e3)
        d3_0=self.d3_0(e4,e3)
        d2_0=self.d2_0(d3_0,e2); d2_1=self.d2_1(d3_0,e2,d2_0)
        d1_0=self.d1_0(d2_0,e1); d1_1=self.d1_1(d2_1,e1,d1_0); d1_2=self.d1_2(d2_1,e1,d1_0,d1_1)
        d0_0=self.d0_0(d1_0,e0); d0_1=self.d0_1(d1_1,e0,d0_0)
        d0_2=self.d0_2(d1_2,e0,d0_0,d0_1); d0_3=self.d0_3(d1_2,e0,d0_0,d0_1,d0_2)
        out=self.head_out(self.up_final(d0_3))
        if self.training and self.ds:
            return [self.head_ds1(d2_1),self.head_ds2(d1_2),self.head_ds3(d0_3),out]
        return out

tmp=UNetPlusPlus(False)
print(f'U-Net++ params: {sum(p.numel() for p in tmp.parameters() if p.requires_grad)/1e6:.1f}M')
del tmp


In [ ]:
# Cell 6: Loss Functions (Tversky for fascicles, BCE+Dice for aponeurosis)
class TverskyLoss(nn.Module):
    def __init__(self,alpha=0.3,beta=0.7,smooth=1e-5):
        super().__init__(); self.a=alpha; self.b=beta; self.s=smooth
    def forward(self,logits,targets):
        p=torch.sigmoid(logits)
        tp=(p*targets).sum(); fp=(p*(1-targets)).sum(); fn=((1-p)*targets).sum()
        return 1.0-(tp+self.s)/(tp+self.a*fp+self.b*fn+self.s)

class MaskedCombinedLoss(nn.Module):
    def __init__(self,apo_pw=3.0,fasc_pw=8.0,smooth=1e-5):
        super().__init__()
        self.apo_pw=apo_pw; self.fasc_pw=fasc_pw; self.smooth=smooth
        self.tversky=TverskyLoss(0.3,0.7)
    def _dice(self,lg,tg):
        p=torch.sigmoid(lg); inter=(p*tg).sum()
        return 1.0-(2*inter+self.smooth)/(p.sum()+tg.sum()+self.smooth)
    def _bce(self,lg,tg,pw):
        return F.binary_cross_entropy_with_logits(lg,tg,
               pos_weight=torch.tensor(pw,device=lg.device))
    def forward(self,logits,targets,flags):
        loss=torch.tensor(0.0,device=logits.device); cnt=0
        for ch in range(2):
            mask=flags[:,ch].bool()
            if not mask.any(): continue
            lg,tg=logits[mask,ch],targets[mask,ch]
            if ch==0: loss=loss+self._bce(lg,tg,self.apo_pw)+self._dice(lg,tg)
            else:     loss=loss+self._bce(lg,tg,self.fasc_pw)+self.tversky(lg,tg)
            cnt+=1
        return loss/max(cnt,1)

def deep_supervision_loss(outputs,targets,flags,criterion):
    total=torch.tensor(0.0,device=outputs[0].device)
    for out in outputs:
        h,w=out.shape[2:]; tgt=F.interpolate(targets,size=(h,w),mode='nearest')
        total=total+criterion(out,tgt,flags)
    return total/len(outputs)

print('Loss functions defined.')


In [ ]:
# Cell 7: Splits & Dataloaders
from sklearn.model_selection import train_test_split
train_all=meta[meta['split']!='test'].copy()
train_all['res_grp']=train_all['orig_w'].astype(str)+'x'+train_all['orig_h'].astype(str)
train_df,val_df=train_test_split(train_all,test_size=0.10,random_state=42,stratify=train_all['res_grp'])
train_df=train_df.reset_index(drop=True); val_df=val_df.reset_index(drop=True)
print(f'Train: {len(train_df)}   Val: {len(val_df)}')
BATCH=8; WORKERS=2
train_ds=MuscleDataset(train_df,aug=True);   val_ds=MuscleDataset(val_df,aug=False)
train_dl=DataLoader(train_ds,batch_size=BATCH,shuffle=True, num_workers=WORKERS,pin_memory=True,drop_last=True)
val_dl  =DataLoader(val_ds,  batch_size=BATCH,shuffle=False,num_workers=WORKERS,pin_memory=True)
print('Dataloaders ready.')


In [ ]:
# Cell 8: Training Loop (40 epochs, OneCycleLR, mixed precision, gradient clipping)
EPOCHS=40; LR=3e-4; CKPT='/kaggle/working/best_unetpp.pth'
model=UNetPlusPlus(deep_supervision=True).to(DEVICE)
criterion=MaskedCombinedLoss(apo_pw=3.0,fasc_pw=8.0)
optimizer=optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
scheduler=optim.lr_scheduler.OneCycleLR(optimizer,max_lr=LR,steps_per_epoch=len(train_dl),
           epochs=EPOCHS,pct_start=0.25,div_factor=10,final_div_factor=100)
scaler=torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

def dice_score(prob,tgt,thresh=0.5):
    p=(prob>thresh).float(); inter=(p*tgt).sum()
    return ((2*inter+1e-5)/(p.sum()+tgt.sum()+1e-5)).item()

best_apo_dice=-1.0
print(f'Training {EPOCHS} epochs | {len(train_dl)} batches/epoch')
for epoch in range(1,EPOCHS+1):
    model.train(); run_loss=0.0
    for imgs,masks,flags in train_dl:
        imgs,masks,flags=imgs.to(DEVICE),masks.to(DEVICE),flags.to(DEVICE)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            outputs=model(imgs)
            loss=deep_supervision_loss(outputs,masks,flags,criterion) if isinstance(outputs,list)                  else criterion(outputs,masks,flags)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer); nn.utils.clip_grad_norm_(model.parameters(),1.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        run_loss+=loss.item()*len(imgs)
    if epoch%5==0 or epoch==1:
        model.eval(); adl,fdl=[],[]
        with torch.no_grad():
            for imgs,masks,flags in val_dl:
                imgs,masks,flags=imgs.to(DEVICE),masks.to(DEVICE),flags.to(DEVICE)
                probs=torch.sigmoid(model(imgs))
                for c,lst in [(0,adl),(1,fdl)]:
                    m=flags[:,c].bool()
                    if m.any(): lst.append(dice_score(probs[m,c],masks[m,c]))
        apo_d=np.mean(adl) if adl else 0.0; fasc_d=np.mean(fdl) if fdl else 0.0
        print(f'Ep {epoch:03d} | Loss {run_loss/len(train_ds):.4f} | Apo={apo_d:.4f} Fasc={fasc_d:.4f} | LR={scheduler.get_last_lr()[0]:.2e}')
        if apo_d>best_apo_dice:
            best_apo_dice=apo_d; torch.save(model.state_dict(),CKPT)
            print(f'  ** Saved (apo={apo_d:.4f})')
print(f'Best apo Dice = {best_apo_dice:.4f}')
model.load_state_dict(torch.load(CKPT,map_location=DEVICE)); model.eval(); print('Checkpoint loaded.')


In [ ]:
# Cell 9: Geometry Engine — FL = MT/sin(PA) avoids line-intersection blow-up
def fit_line_huber(pts_yx):
    if len(pts_yx)<15: return None
    pts=pts_yx[:,::-1].astype(np.float32)
    line=cv2.fitLine(pts.reshape(-1,1,2),cv2.DIST_HUBER,0,0.01,0.01)
    vx,vy,x0,y0=line.flatten()
    return float(vx),float(vy),float(x0),float(y0)

def to_mb(vx,vy,x0,y0):
    if abs(vx)<1e-9: return None
    m=vy/vx; return float(m),float(y0-m*x0)

def perp_dist(m_d,b_d,m_s,b_s,x_lo,x_hi,n=200):
    xs=np.linspace(x_lo,x_hi,n)
    return float(np.abs(m_d*xs+b_d-m_s*xs-b_s).mean()/math.sqrt(m_d**2+1))

def signed_pa(tf,ta):
    d=tf-ta
    while d>90:  d-=180
    while d<-90: d+=180
    return abs(d)

def measure_image(apo_hw, fasc_hw, orig_w, orig_h):
    apo_full =cv2.resize(apo_hw, (orig_w,orig_h),interpolation=cv2.INTER_LINEAR)
    fasc_full=cv2.resize(fasc_hw,(orig_w,orig_h),interpolation=cv2.INTER_LINEAR)
    apo_bin =(apo_full >0.40).astype(np.uint8)
    fasc_bin=(fasc_full>0.35).astype(np.uint8)

    lbl,n=ndimage.label(apo_bin); apo_lines=[]
    if n>=2:
        sizes=[(lbl==i).sum() for i in range(1,n+1)]
        top2=sorted(range(1,n+1),key=lambda i:sizes[i-1],reverse=True)[:2]
        top2=sorted(top2,key=lambda i:np.where(lbl==i)[0].mean())
        for lb in top2:
            yx=np.column_stack(np.where(lbl==lb)).astype(np.float32)
            lf=fit_line_huber(yx)
            if lf: apo_lines.append(to_mb(*lf))
    elif n==1:
        ys,xs=np.where(apo_bin>0); mid_y=ys.mean()
        for sel in [ys<mid_y,ys>=mid_y]:
            if sel.sum()>=15:
                yx=np.column_stack((ys[sel],xs[sel])).astype(np.float32)
                lf=fit_line_huber(yx)
                if lf: apo_lines.append(to_mb(*lf))

    if len(apo_lines)<2 or None in apo_lines: return None
    sup_m,sup_b=apo_lines[0]; dep_m,dep_b=apo_lines[1]
    theta_dep=math.degrees(math.atan(dep_m))
    x_lo,x_hi=int(orig_w*0.05),int(orig_w*0.95)
    mt_px=perp_dist(dep_m,dep_b,sup_m,sup_b,x_lo,x_hi)
    if mt_px<5 or mt_px>orig_h*0.8: return None

    f_lbl,n_fasc=ndimage.label(fasc_bin); pa_list,fl_list=[],[]
    for lb in range(1,n_fasc+1):
        yx=np.column_stack(np.where(f_lbl==lb)).astype(np.float32)
        if len(yx)<15: continue
        lf=fit_line_huber(yx)
        if not lf: continue
        mb=to_mb(*lf)
        if not mb: continue
        fm,_=mb; pa=signed_pa(math.degrees(math.atan(fm)),theta_dep)
        if pa<2.0 or pa>45.0: continue
        fl_px=mt_px/math.sin(math.radians(pa))   # SAFE triangle formula
        if fl_px<mt_px or fl_px>orig_w*3: continue
        pa_list.append(pa); fl_list.append(fl_px)

    apo_frac=float(apo_bin.mean()); rel=0.0
    if 0.001<apo_frac<0.35: rel+=0.4
    if len(pa_list)>=2:     rel+=0.4
    elif len(pa_list)==1:   rel+=0.2
    if 5<mt_px<orig_h*0.5: rel+=0.2

    return dict(pa_deg=float(np.median(pa_list)) if pa_list else None,
                fl_px =float(np.median(fl_list))  if fl_list else None,
                mt_px=mt_px,reliability=rel,n_fasc=len(pa_list))

print('Geometry engine ready.')


In [ ]:
# Cell 10: Per-Resolution Fallback Medians from GT Masks
print('Computing per-resolution medians from GT masks...')
dual_rows=meta[meta['split']=='train_dual'].sample(min(500,(meta['split']=='train_dual').sum()),random_state=0)
res_records=[]
for _,row in dual_rows.iterrows():
    fname=row['image_id']; orig_w=int(row['orig_w']); orig_h=int(row['orig_h'])
    stem=os.path.splitext(fname)[0]; mmpx=get_mmpx(stem,orig_w,orig_h)
    try:
        a=load_mask_np(os.path.join(APO_MSK_DIR, fname))
        f=load_mask_np(os.path.join(FASC_MSK_DIR,fname))
        r=measure_image(a,f,orig_w,orig_h)
    except: continue
    if r is None or r['pa_deg'] is None: continue
    pa=r['pa_deg']; mt=r['mt_px']*mmpx
    fl=r['fl_px']*mmpx if r['fl_px'] else mt/math.sin(math.radians(pa))
    if not (2<=pa<=45 and 3<=mt<=60 and 20<=fl<=250): continue
    res_records.append(dict(res=f'{orig_w}x{orig_h}',pa_deg=pa,fl_mm=fl,mt_mm=mt))

gt_df=pd.DataFrame(res_records); print(f'Usable images: {len(gt_df)}')
GLOBAL_PA=float(gt_df['pa_deg'].median()) if len(gt_df) else 14.87
GLOBAL_FL=float(gt_df['fl_mm'].median())  if len(gt_df) else 91.38
GLOBAL_MT=float(gt_df['mt_mm'].median())  if len(gt_df) else 18.34
print(f'Global fallback: PA={GLOBAL_PA:.2f} FL={GLOBAL_FL:.2f} MT={GLOBAL_MT:.2f}')
RES_MEDIANS={}
if len(gt_df):
    for res,g in gt_df.groupby('res'):
        RES_MEDIANS[res]={'pa':g['pa_deg'].median(),'fl':g['fl_mm'].median(),'mt':g['mt_mm'].median()}
    for res,v in sorted(RES_MEDIANS.items()):
        print(f'  {res}: PA={v["pa"]:.1f} FL={v["fl"]:.1f} MT={v["mt"]:.1f}')


In [ ]:
# Cell 11: Proxy Validation (GT masks vs model with TTA)
print('Proxy MAE on val split...')
model.eval(); gt_pa,gt_fl,gt_mt,pr_pa,pr_fl,pr_mt=[],[],[],[],[],[]
val_dual=val_df[val_df['has_apo']&val_df['has_fasc']].reset_index(drop=True)
for _,row in val_dual.iterrows():
    fname=row['image_id']; orig_w=int(row['orig_w']); orig_h=int(row['orig_h'])
    stem=os.path.splitext(fname)[0]; mmpx=get_mmpx(stem,orig_w,orig_h)
    try:
        a_gt=load_mask_np(os.path.join(APO_MSK_DIR, fname))
        f_gt=load_mask_np(os.path.join(FASC_MSK_DIR,fname))
        res_gt=measure_image(a_gt,f_gt,orig_w,orig_h)
    except: res_gt=None
    img_np=load_img_np(os.path.join(FASC_IMG_DIR,fname))
    img_t=torch.tensor(img_np).unsqueeze(0).unsqueeze(0).to(DEVICE)
    img_tf=torch.flip(img_t,dims=[3])
    with torch.no_grad():
        p1=torch.sigmoid(model(img_t))[0].cpu().numpy()
        p2=torch.sigmoid(model(img_tf))[0].cpu().numpy()
        p2=p2[:,:,::-1].copy(); prob=(p1+p2)/2.0
    try: res_pr=measure_image(prob[0],prob[1],orig_w,orig_h)
    except: res_pr=None
    if res_gt is None or res_pr is None: continue
    if res_gt['pa_deg'] is None or res_pr['pa_deg'] is None: continue
    gt_pa.append(res_gt['pa_deg']); gt_mt.append(res_gt['mt_px']*mmpx)
    gt_fl.append((res_gt['fl_px'] or res_gt['mt_px']/math.sin(math.radians(res_gt['pa_deg'])))*mmpx)
    pr_pa.append(res_pr['pa_deg']); pr_mt.append(res_pr['mt_px']*mmpx)
    pr_fl.append((res_pr['fl_px'] or res_pr['mt_px']/math.sin(math.radians(res_pr['pa_deg'])))*mmpx)
if gt_pa:
    print(f'Proxy MAE on {len(gt_pa)} val images:')
    print(f'  PA: {np.mean(np.abs(np.array(gt_pa)-np.array(pr_pa))):.3f} deg')
    print(f'  FL: {np.mean(np.abs(np.array(gt_fl)-np.array(pr_fl))):.3f} mm')
    print(f'  MT: {np.mean(np.abs(np.array(gt_mt)-np.array(pr_mt))):.3f} mm')
else:
    print('Not enough paired results.')


In [ ]:
# Cell 12: Test Inference + Confidence-Weighted Blend + Submission
def get_fallback(res_grp):
    if res_grp in RES_MEDIANS:
        m=RES_MEDIANS[res_grp]; return m['pa'],m['fl'],m['mt']
    return GLOBAL_PA,GLOBAL_FL,GLOBAL_MT

def blend(geo_val,med_val,reliability,scale_from_csv):
    w=reliability*(1.0 if scale_from_csv else 0.7)
    return w*geo_val+(1.0-w)*med_val

test_sub=meta[meta['split']=='test'].reset_index(drop=True)
rows=[]; n_fallback=0; print(f'Inference on {len(test_sub)} test images...')
model.eval()

for _,row in test_sub.iterrows():
    fname=row['image_id']; orig_w=int(row['orig_w']); orig_h=int(row['orig_h'])
    stem=os.path.splitext(fname)[0]; mmpx=get_mmpx(stem,orig_w,orig_h)
    res_grp=f'{orig_w}x{orig_h}'; fb_pa,fb_fl,fb_mt=get_fallback(res_grp)
    scale_from_csv=isinstance(SCALE_MAP.get(stem),float)
    img_np=load_img_np(os.path.join(TEST_IMG_DIR,fname))
    img_t=torch.tensor(img_np).unsqueeze(0).unsqueeze(0).to(DEVICE)
    img_tf=torch.flip(img_t,dims=[3])
    with torch.no_grad():
        p1=torch.sigmoid(model(img_t))[0].cpu().numpy()
        p2=torch.sigmoid(model(img_tf))[0].cpu().numpy()
        p2=p2[:,:,::-1].copy(); prob=(p1+p2)/2.0
    try: result=measure_image(prob[0],prob[1],orig_w,orig_h)
    except: result=None
    if result and result['pa_deg'] is not None:
        rel=result['reliability']; pa_geo=result['pa_deg']; mt_geo=result['mt_px']*mmpx
        fl_geo=(result['fl_px']*mmpx) if result['fl_px'] else mt_geo/math.sin(math.radians(pa_geo))
        pa=blend(pa_geo,fb_pa,rel,scale_from_csv)
        fl=blend(fl_geo,fb_fl,rel,scale_from_csv)
        mt=blend(mt_geo,fb_mt,rel,scale_from_csv)
    else:
        pa,fl,mt=fb_pa,fb_fl,fb_mt; n_fallback+=1
    pa=float(np.clip(pa, 3.0,45.0)); fl=float(np.clip(fl,15.0,250.0)); mt=float(np.clip(mt,5.0,60.0))
    rows.append({'image_id':fname,'pa_deg':round(pa,3),'fl_mm':round(fl,3),'mt_mm':round(mt,3)})

sub=pd.DataFrame(rows)
sub.to_csv('/kaggle/working/submission.csv',index=False,sep=',')
print(f'[SUCCESS] submission.csv: {len(sub)} rows')
print(f'Geometry used: {len(sub)-n_fallback} ({(len(sub)-n_fallback)/len(sub)*100:.1f}%) | Full fallback: {n_fallback}')
print(sub[['pa_deg','fl_mm','mt_mm']].describe())
with open('/kaggle/working/submission.csv') as ff:
    for _ in range(5): print(ff.readline().rstrip())


In [ ]:
# Cell 13: Median-Only Baseline (safe backup)
base_rows=[]
for _,row in test_sub.iterrows():
    fname=row['image_id']; orig_w=int(row['orig_w']); orig_h=int(row['orig_h'])
    pa,fl,mt=get_fallback(f'{orig_w}x{orig_h}')
    base_rows.append({'image_id':fname,'pa_deg':round(pa,3),'fl_mm':round(fl,3),'mt_mm':round(mt,3)})
base_sub=pd.DataFrame(base_rows)
base_sub.to_csv('/kaggle/working/submission_median_baseline.csv',index=False,sep=',')
print(f'Median baseline: {len(base_sub)} rows written.')
print(base_sub[['pa_deg','fl_mm','mt_mm']].describe())
